<html><div style="font-size:7pt">This notebook may contain text, code and images generated by artificial intelligence. Used model: claude-sonnet-4-20250514, vision model: claude-sonnet-4-20250514, endpoint: None, bia-bob version: 0.34.3.. It is good scientific practice to check the code and results it produces carefully. <a href="https://github.com/haesleinhuepf/bia-bob">Read more about code generation using bia-bob</a></div></html>

# Image Embeddings and UMAP Generation

This notebook demonstrates how to:
1. Generate vision embeddings using the [openai/clip-vit-base-patch32](https://huggingface.co/openai/clip-vit-base-patch32)
2. Create a 3D UMAP visualization of the embeddings
3. Save the results for VEST

In [1]:
import os
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn.functional as F
from transformers import AutoImageProcessor, AutoModel
from datasets import load_dataset
from umap import UMAP
import random
from pathlib import Path
import requests
from io import BytesIO
import torch

In [2]:
import transformers
transformers.__version__

'4.56.1'

In [3]:
base_dir = Path("./")
images_dir = base_dir / "logos"

## Load Vision Embedding Model

In [4]:
from transformers import CLIPProcessor, CLIPModel
import pandas as pd
import torch
from PIL import Image
import requests
import os

# Initialize models
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

## Generate Vision Embeddings for All Images

In [5]:
import os
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn.functional as F

# Get list of all image files in the images folder and subfolders
image_files = []
for root, dirs, files in os.walk(images_dir):
    for f in files:
        if f.lower().endswith('.png') or f.lower().endswith('.jpg'):
            image_files.append(os.path.join(root, f))

# Initialize lists to store results
filenames = []
embeddings = []
images = []

print(f"{len(image_files)} images found")

199 images found


In [7]:
#image_files = random.sample(image_files, 1000)

In [8]:
print(f"Processing {len(image_files)} images for embeddings...")

# Loop through all image files
for i, image_path in enumerate(image_files):
    # Obtain filename from path
    filename = os.path.relpath(image_path, images_dir)
    
    # Load the image
    current_image = Image.open(image_path)
    images.append(np.asarray(current_image))

    # Apply the processing pipeline
    current_inputs = clip_processor(images=current_image, return_tensors="pt")
    with torch.no_grad():
        current_np_emb = clip_model.get_image_features(**current_inputs).squeeze().tolist()

    # Store results
    filenames.append(filename)
    embeddings.append(current_np_emb)

    if (i + 1) % 100 == 0:
        print(f"Processed {i + 1}/{len(image_files)} images")



# Create DataFrame
df = pd.DataFrame({
    'filename': filenames,
    'embedding': embeddings
})

print(f"Successfully processed {len(df)} images")
display(df.head())

Processing 199 images for embeddings...
Processed 100/199 images
Successfully processed 199 images


,filename,embedding
0,01_codeloom.png,"[0.3220980167388916, 0.24343657493591309, 0.37..."
1,02_devnexus.png,"[0.14768806099891663, -0.20853744447231293, -0..."
2,03_bitsprint.png,"[0.1686640828847885, -0.08660081028938293, -0...."
3,04_appvanta.png,"[-0.3357119560241699, 0.004972964525222778, 0...."
4,05_stackforge.png,"[0.23733238875865936, -0.6277705430984497, -0...."


## Create 3D UMAP Visualization from Embeddings

In [9]:
# Convert embeddings list to numpy array matrix
embedding_matrix = np.stack(df['embedding'].values)

print(f"Embedding matrix shape: {embedding_matrix.shape}")
print("Creating 3D UMAP coordinates...")

# Create 3D UMAP
umap_reducer = UMAP(n_components=3, random_state=42)
umap_coords_actual = umap_reducer.fit_transform(embedding_matrix)

# Add UMAP coordinates to dataframe
df['x'] = umap_coords_actual[:, 0]
df['y'] = umap_coords_actual[:, 1]
df['z'] = umap_coords_actual[:, 2]

print("UMAP coordinates created successfully")
print(f"X range: {df['x'].min():.2f} to {df['x'].max():.2f}")
print(f"Y range: {df['y'].min():.2f} to {df['y'].max():.2f}")
print(f"Z range: {df['z'].min():.2f} to {df['z'].max():.2f}")
display(df.head())

Embedding matrix shape: (199, 512)
Creating 3D UMAP coordinates...


C:\Users\rober\miniforge3\envs\bob-env\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP coordinates created successfully
X range: 9.00 to 13.03
Y range: 1.88 to 5.14
Z range: 7.30 to 10.32


,filename,embedding,x,y,z
0,01_codeloom.png,"[0.3220980167388916, 0.24343657493591309, 0.37...",12.333786,4.766936,9.541565
1,02_devnexus.png,"[0.14768806099891663, -0.20853744447231293, -0...",10.755573,2.410110,8.194623
2,03_bitsprint.png,"[0.1686640828847885, -0.08660081028938293, -0....",11.804419,3.454775,7.366132
3,04_appvanta.png,"[-0.3357119560241699, 0.004972964525222778, 0....",10.904742,5.010376,8.371152
4,05_stackforge.png,"[0.23733238875865936, -0.6277705430984497, -0....",12.118531,4.474287,8.896679


## Save Results to CSV File

In [10]:
# Save the dataframe to CSV
output_path = base_dir / "data.csv"

# Convert embedding arrays to strings for CSV storage
df_to_save = df.copy()
df_to_save['embedding'] = df_to_save['embedding'].apply(lambda x: ','.join(map(str, x)))

df_to_save.to_csv(output_path, index=False)

print(f"Results saved to {output_path}")
print(f"Final dataframe shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
display(df[['filename', 'x', 'y', 'z']].head(10))

Results saved to data.csv
Final dataframe shape: (199, 5)
Columns: ['filename', 'embedding', 'x', 'y', 'z']


,filename,x,y,z
0,01_codeloom.png,12.333786,4.766936,9.541565
1,02_devnexus.png,10.755573,2.410110,8.194623
2,03_bitsprint.png,11.804419,3.454775,7.366132
3,04_appvanta.png,10.904742,5.010376,8.371152
4,05_stackforge.png,12.118531,4.474287,8.896679
5,06_pixelroot.png,10.537862,2.261901,9.339557
6,07_nexasoft.png,10.133633,1.983567,8.392517
7,08_echostream.png,11.299736,3.704078,7.488616
8,09_logicpulse.png,11.826912,4.213936,9.613666
9,100_chemvanta.png,11.609924,3.777063,9.737745
